# BLACKBOX × IBM Granite — advisory certificate narrative

Runs **on the LinuxONE (s390x) Jupyter container**. Pairs the structured
Decision Certificate facts with IBM Granite reasoning.

**Advisory only** — scores and gate decisions never depend on this text.
Model: `ibm-granite/granite-3.0-1b-a400m-instruct` (Apache-2.0), cached in
`~/shared/hf_cache`. Greedy decoding (deterministic).

In [ ]:
# Certificate facts as returned by POST /explain on the guest (interaction-aware sklearn).
# Replaced at demo time by a live fetch when the BLACKBOX API is reachable.
summary = {
    "transaction_id": "TX-DEMO-0001",
    "impact_score": 0.516452,
    "gate_decision": False,
    "status": "NO_SIMULATION",
    "policy_formula": "I=0.45*A+0.30*W+0.15*D+0.10*L;threshold=0.60;mode=interaction_aware",
    "backend": "sklearn",
    "hybrid_risk": "on",
    "risk_model": "HybridTreeEnsemble (RF trees + XGBoost models) [scikit-learn-1.9.1 + xgboost-3.4.1]",
}
print(summary)

In [ ]:
import json, os, time
os.environ.setdefault("HF_HOME", "/home/ibm-user/shared/hf_cache")
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL = "ibm-granite/granite-3.0-1b-a400m-instruct"
system = (
    "You explain ML-gate decision certificates to payment-operations staff. "
    "Use ONLY the facts provided in the certificate summary. Be concise "
    "(max 120 words). Never invent numbers or outcomes. If a fact is "
    "missing or unknown, say so explicitly."
)
prompt = (
    f"{system}\n\nCertificate facts (JSON):\n"
    f"{json.dumps(summary, sort_keys=True)}\n\n"
    "Write the 3-4 sentence narrative an operator should read first."
)
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32)
model.eval()
inputs = tok(prompt, return_tensors="pt")
t0 = time.time()
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
text = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(f"latency: {time.time()-t0:.1f}s (CPU, s390x)\n")
print(text.strip())
print("\n[advisory only — never changes scores or decisions]")